# 01 — Preprocessing CIC-IDS2017 Versi 2

**Masalah pada versi 1** (lihat `05_Data_Eksperimen/CIC-IDS2017/arsip_hasil_v1/`):
1. Dataset tidak punya kolom IP sumber dan waktu, sehingga **fitur agregasi tidak bisa dihitung**.
2. Duplikat dibuang dengan membandingkan fitur saja. Aliran port scan dan brute force SSH
   memang mirip satu sama lain, sehingga **43–45% aliran kedua serangan itu ikut terbuang**,
   padahal ciri utamanya justru *jumlahnya yang banyak*.

**Perbaikan pada versi 2:**
- Memakai versi dataset yang memuat Source IP dan Timestamp.
- Fitur agregasi dihitung **paling awal**, dari seluruh aliran, sebelum ada baris yang dibuang.
- Duplikat hanya dibuang bila **seluruh kolom** sama (termasuk Flow ID, port sumber, dan waktu).
- Fitur aliran = **68 fitur yang sama persis** dengan versi 1, supaya hasilnya bisa dibandingkan.
- Fungsi agregasi diambil dari `04_Kode/nids-deteksi/nids/agregasi.py`, **fungsi yang sama**
  dengan yang dipakai pada data server, jadi tidak ada dua versi rumus.

| Langkah | Isi |
|---|---|
| 1 | Muat 8 berkas, buang baris kosong, rapikan label |
| 2 | Cocokkan setiap baris dengan versi v1 (bukti keaslian mirror) |
| 3 | Perbaiki kolom waktu |
| 4 | Hitung 7 fitur agregasi untuk jendela 60, 120, dan 300 detik |
| 5 | Buang baris inf/NaN dan nilai negatif tidak valid (aturan sama dengan v1) |
| 6 | Buang duplikat persis |
| 7 | Bagi data: latih = Senin (normal), validasi 20% / uji 80% Selasa–Jumat |
| 8 | Simpan ke `data/processed_v2/` + laporan `laporan_preprocessing_v2.md` |

In [ ]:
import os, sys

# Lokasi kerja: Google Colab (Drive) atau laptop
try:
    from google.colab import drive
    drive.mount('/content/drive')
    FOLDER = '/content/drive/MyDrive/TA/CIC-IDS2017'          # folder notebook ini di Google Drive
    CALON_KODE = [os.path.join(FOLDER, '2_sistem_nids'),      # lokasi folder kode sistem NIDS yang dicari
                  '/content/drive/MyDrive/TA/2_sistem_nids',
                  '/content/drive/MyDrive/TA/nids-deteksi',
                  '/content/drive/MyDrive/TA/04_Kode/nids-deteksi']
except ImportError:
    FOLDER = os.path.abspath(os.path.join('..', '..', '05_Data_Eksperimen', 'CIC-IDS2017'))  # folder data (notebook ada di 04_Kode/4_notebook)
    CALON_KODE = [os.path.abspath(os.path.join('..', '2_sistem_nids'))]                       # kode sistem NIDS

KODE = next((p for p in CALON_KODE if os.path.isdir(os.path.join(p, 'nids'))), None)
os.makedirs(FOLDER, exist_ok=True)
os.chdir(FOLDER)
DATA = os.path.join(FOLDER, 'data')
print('Folder kerja :', FOLDER)
print('Kode nids    :', KODE or 'tidak ditemukan (hanya dibutuhkan notebook 01)')

In [ ]:
import json, time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

if KODE is None:
    raise FileNotFoundError(
        'Folder nids-deteksi tidak ditemukan. Di Google Colab: unggah folder '
        'D:/KULIAH/TA/04_Kode/nids-deteksi ke Google Drive, ke MyDrive/TA/nids-deteksi, '
        'lalu jalankan ulang notebook dari sel pertama.')
sys.path.insert(0, KODE)
from nids.agregasi import FITUR_AGREGASI, tambah_agregasi   # fungsi yang sama dengan pipeline server

RAW, RAW_V1 = os.path.join(DATA, 'raw_labelled'), os.path.join(DATA, 'raw')
ADA_V1 = all(os.path.exists(os.path.join(RAW_V1, fn)) for fn in [
    'Monday-WorkingHours.pcap_ISCX.csv', 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'])
print('Data versi 1 untuk pencocokan:', 'ada' if ADA_V1 else 'tidak ada (pencocokan dilewati)')
OUT = os.path.join(DATA, 'processed_v2')
os.makedirs(OUT, exist_ok=True)
SEED = 42
JENDELA = [60, 120, 300]

FILES = [
    ('Senin',          'Monday-WorkingHours.pcap_ISCX.csv'),
    ('Selasa',         'Tuesday-WorkingHours.pcap_ISCX.csv'),
    ('Rabu',           'Wednesday-workingHours.pcap_ISCX.csv'),
    ('Kamis pagi',     'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv'),
    ('Kamis siang',    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv'),
    ('Jumat pagi',     'Friday-WorkingHours-Morning.pcap_ISCX.csv'),
    ('Jumat siang-PS', 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv'),
    ('Jumat siang-DD', 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'),
]
# pemetaan label CIC-IDS2017 ke skenario serangan TA
SKENARIO = {'BENIGN': 'normal', 'SSH-Patator': 'ssh_bruteforce', 'PortScan': 'port_scan',
            'DoS Hulk': 'http_dos', 'DoS GoldenEye': 'http_dos', 'DoS slowloris': 'http_dos',
            'DoS Slowhttptest': 'http_dos', 'DDoS': 'http_dos'}
# -1 pada dua kolom ini = "tidak ada paket pada arah itu", bukan kesalahan
SENTINEL_NEG = {'Init_Win_bytes_forward', 'Init_Win_bytes_backward'}
FITUR68 = json.load(open(os.path.join(FOLDER, 'fitur_68.json'), encoding='utf-8'))['fitur']   # sama dengan v1
META = ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol',
        'Waktu', 'Hari', 'Label', 'Label_biner', 'Skenario']

langkah = []
def catat(tahap, sebelum, sesudah, ket=''):
    langkah.append((tahap, sebelum, sesudah, ket))
    print('  %-44s %10s -> %10s  (-%s) %s' % (tahap, f'{sebelum:,}', f'{sesudah:,}', f'{sebelum - sesudah:,}', ket))

def rapikan_label(s):
    return (s.astype(str).str.replace('\x96', '-', regex=False).str.replace('\ufffd', '-', regex=False)
             .str.replace('\u00ef\u00bf\u00bd', '-', regex=False).str.replace(r'\s+', ' ', regex=True).str.strip())
t0 = time.time()

## Langkah 1–2: Muat data dan cocokkan dengan versi 1

Setiap baris versi 2 dibandingkan dengan baris yang sama di versi 1 (semua kolom fitur).
Berkas Senin versi 1 dibulatkan ke 10 desimal, jadi dipakai toleransi pembulatan 10⁻⁸.
Bila hasilnya 100%, berarti mirror ini **berisi data yang sama** dengan dataset yang dipakai sebelumnya.
Langkah ini dilewati bila data versi 1 (`data/raw/`) tidak tersedia, misalnya di Google Colab.
Hasil pencocokan di laptop: **100% identik untuk semua hari** (lihat `laporan_preprocessing_v2.md`).

In [ ]:
parts, per_file, cocok = [], [], []
for hari, fn in FILES:
    df = pd.read_csv(os.path.join(RAW, fn), encoding='latin-1', low_memory=False)
    df.columns = df.columns.str.strip()
    n_mentah = len(df)
    df = df.dropna(how='all').reset_index(drop=True)          # baris kosong di akhir berkas Kamis pagi
    if ADA_V1:
        v1 = pd.read_csv(os.path.join(RAW_V1, fn), encoding='latin-1', low_memory=False)
        v1.columns = v1.columns.str.strip()
        kol = [c for c in v1.columns if c != 'Label' and not c.endswith('.1')]
        a = df[kol].apply(pd.to_numeric, errors='coerce').to_numpy(np.float64)
        b = v1[kol].apply(pd.to_numeric, errors='coerce').to_numpy(np.float64)
        sama = float(np.isclose(a, b, rtol=1e-8, atol=0, equal_nan=True).all(axis=1).mean()) if len(a) == len(b) else 0.0
        cocok.append((hari, len(df), len(v1), sama))
        del v1, a, b
    df['Hari'] = hari
    per_file.append((hari, fn, n_mentah, len(df)))
    parts.append(df)
df = pd.concat(parts, ignore_index=True)
del parts
n_raw = len(df)
df['Label'] = rapikan_label(df['Label'])
label_mentah = df['Label'].value_counts()                     # sebelum ada baris yang dibuang
df = df.drop(columns=[c for c in df.columns if c.endswith('.1')])          # kolom kembar
NUM = [c for c in df.columns if c not in ('Flow ID', 'Source IP', 'Destination IP', 'Timestamp', 'Label', 'Hari')]
df[NUM] = df[NUM].apply(pd.to_numeric, errors='coerce')

print('Total: %s aliran' % f'{n_raw:,}')
if cocok:
    display(pd.DataFrame(cocok, columns=['Hari', 'Baris v2', 'Baris v1', 'Identik']).style.format({'Identik': '{:.2%}'}))

## Langkah 3: Perbaiki kolom waktu

Dua keanehan pada kolom `Timestamp` CIC-IDS2017:
- Format **12 jam tanpa AM/PM**. Perekaman berlangsung 08.40–17.10, jadi jam 1–7 berarti siang (13.00–19.00).
- Senin tercatat sampai **detik**, Selasa–Jumat hanya sampai **menit**. Agar perlakuannya seragam,
  semua waktu dibulatkan ke bawah ke menit. Akibatnya, jendela 60 detik = aliran dalam menit yang sama.

In [ ]:
t = pd.to_datetime(df['Timestamp'].astype(str).str.strip(), dayfirst=True, format='mixed')
t = t + pd.to_timedelta(np.where(t.dt.hour < 8, 12, 0), unit='h')
df['Waktu'] = t.dt.floor('min')
rentang = df.groupby('Hari', sort=False)['Waktu'].agg(['min', 'max'])
rentang

## Langkah 4: Fitur agregasi jendela waktu

Untuk setiap aliran dihitung perilaku sumbernya dalam W detik **terakhir** (hanya melihat ke belakang,
sama seperti saat deteksi langsung):

| Fitur | Arti | Pola yang ditangkap |
|---|---|---|
| `agg_ip_aliran` | jumlah aliran dari IP sumber yang sama | brute force, flood |
| `agg_ip_port_unik` | jumlah port tujuan berbeda dari IP itu | port scan |
| `agg_ip_rasio_gagal` | proporsi aliran yang tidak dibalas / ditolak | port scan, brute force |
| `agg_ip_aliran_ssh` | jumlah aliran ke port 22 | brute force SSH |
| `agg_s24_ip_unik` | jumlah IP berbeda dari subnet /24 yang sama | pemindaian terdistribusi |
| `agg_s24_aliran` | jumlah aliran dari subnet /24 itu | pemindaian terdistribusi |
| `agg_s24_port_unik` | jumlah port tujuan berbeda dari subnet /24 itu | pemindaian terdistribusi |

Dihitung dari **seluruh** aliran sebelum pembersihan, untuk 3 panjang jendela. Jendela terbaik dipilih nanti di notebook 02.

In [ ]:
src = df[['Source IP', 'Destination Port', 'Total Backward Packets',
          'Total Length of Bwd Packets', 'RST Flag Count']].copy()
src['Timestamp'] = df['Waktu'].dt.tz_localize('UTC')
src = src.fillna({'Total Backward Packets': 0, 'Total Length of Bwd Packets': 0, 'RST Flag Count': 0})
kolom_agg = {}
for w in JENDELA:
    t1 = time.time()
    h = tambah_agregasi(src, jendela_detik=w)
    for c in FITUR_AGREGASI:
        kolom_agg['%s_w%d' % (c, w)] = h[c].to_numpy(np.float32)
    print('jendela %3d detik selesai (%.0f detik)' % (w, time.time() - t1))
AGG_KOL = list(kolom_agg)
df = pd.concat([df, pd.DataFrame(kolom_agg, index=df.index)], axis=1)
del src, h, kolom_agg

## Langkah 5–6: Pembersihan

Aturan sama dengan versi 1 (inf/NaN, nilai negatif yang mustahil), **kecuali** cara membuang duplikat.

In [ ]:
n0 = len(df)
df[FITUR68] = df[FITUR68].replace([np.inf, -np.inf], np.nan)
nan_cols = df[FITUR68].isna().sum()
nan_cols = nan_cols[nan_cols > 0]
df = df.dropna(subset=FITUR68)
catat('Buang baris dengan inf / NaN', n0, len(df), 'kolom: ' + ', '.join(nan_cols.index))

cek_neg = [c for c in FITUR68 if c not in SENTINEL_NEG]
neg = df[cek_neg] < 0
neg_cols = neg.sum()
neg_cols = neg_cols[neg_cols > 0].sort_values(ascending=False)
n0 = len(df)
df = df[~neg.any(axis=1)]
del neg
catat('Buang baris bernilai negatif tidak valid', n0, len(df), '%d kolom terdampak' % len(neg_cols))

n0 = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c not in AGG_KOL + ['Waktu', 'Hari']])
catat('Buang duplikat persis (semua kolom sama)', n0, len(df))

konstan = [c for c in FITUR68 if df[c].nunique() <= 1]
print('Fitur konstan:', konstan or 'tidak ada')

### Jumlah aliran per label: mentah vs versi 1 vs versi 2
Perhatikan baris **PortScan** dan **SSH-Patator**: versi 2 mempertahankan hampir semuanya.

In [ ]:
df['Label_biner'] = (df['Label'] != 'BENIGN').astype('int8')
df['Skenario'] = df['Label'].map(SKENARIO).fillna('lainnya')
df[FITUR68] = df[FITUR68].astype('float32')
for c in ('Source Port', 'Destination Port', 'Protocol'):
    df[c] = df[c].astype('int32')

label_v2 = df['Label'].value_counts()
kolom = {'Mentah': label_mentah}
P1 = os.path.join(DATA, 'processed_v1')
if os.path.exists(os.path.join(P1, 'test.csv.gz')):
    kolom['Versi 1'] = pd.concat([pd.read_csv(os.path.join(P1, n + '.csv.gz'), usecols=['Label'])['Label']
                                  for n in ('train_senin_normal', 'val', 'test')]).value_counts()
else:   # angka versi 1 dari laporan_preprocessing.md (arsip_hasil_v1)
    kolom['Versi 1'] = pd.Series({'BENIGN': 2092335, 'DoS Hulk': 172688, 'DDoS': 127995, 'PortScan': 90694,
                                  'DoS GoldenEye': 10281, 'FTP-Patator': 5927, 'DoS slowloris': 5385,
                                  'DoS Slowhttptest': 5228, 'SSH-Patator': 3217, 'Bot': 1948,
                                  'Web Attack - Brute Force': 1470, 'Web Attack - XSS': 652, 'Infiltration': 35,
                                  'Web Attack - Sql Injection': 21, 'Heartbleed': 7})
kolom['Versi 2'] = label_v2
tabel_label = pd.DataFrame(kolom).fillna(0).astype(int).sort_values('Mentah', ascending=False)
tabel_label

## Langkah 7: Pembagian data

- **Latih** = seluruh lalu lintas normal hari **Senin** (CIC-IDS2017 merekam Senin tanpa serangan).
  Ini setara dengan *baseline* normal yang pada sistem nyata diambil dari server.
- **Validasi** = 20% Selasa–Jumat (stratified per label), untuk memilih contamination dan panjang jendela.
- **Uji** = 80% sisanya, dipakai **sekali** untuk pelaporan akhir.

In [ ]:
train = df[(df['Hari'] == 'Senin') & (df['Label_biner'] == 0)]
rest = df[df['Hari'] != 'Senin']
val, test = train_test_split(rest, test_size=0.80, stratify=rest['Label'], random_state=SEED)
del rest
pembagian = pd.DataFrame([(n, len(d), int((d.Label_biner == 0).sum()), int((d.Label_biner == 1).sum()))
                          for n, d in (('Latih (Senin)', train), ('Validasi', val), ('Uji', test))],
                         columns=['Subset', 'Total', 'Normal', 'Serangan'])
pembagian

## Langkah 8: Simpan hasil dan laporan

In [ ]:
for nama, d in (('train_senin_normal', train), ('val', val), ('test', test)):
    p = os.path.join(OUT, nama + '.parquet')
    d[META + FITUR68 + AGG_KOL].to_parquet(p, index=False)
    print('  %-28s %8.1f MB' % (nama + '.parquet', os.path.getsize(p) / 1048576))
json.dump({'fitur_aliran': FITUR68, 'fitur_agregasi': FITUR_AGREGASI, 'jendela_detik': JENDELA,
           'kolom_agregasi': AGG_KOL, 'kolom_metadata': META},
          open(os.path.join(OUT, 'fitur_v2.json'), 'w', encoding='utf-8'), indent=2, ensure_ascii=False)

def angka(n):
    return f'{n:,}'.replace(',', '.')

def tabel(header, rows):
    s = '| ' + ' | '.join(header) + ' |\n|' + '|'.join('---' for _ in header) + '|\n'
    return s + ''.join('| ' + ' | '.join(str(x) for x in r) + ' |\n' for r in rows)

penyerang = df.loc[df.Label_biner == 1, 'Source IP'].value_counts()
md = ['# Laporan Preprocessing CIC-IDS2017 — Versi 2\n',
      'Dibuat otomatis oleh `01_Preprocessing_v2.ipynb`. Waktu proses: %d detik.\n' % (time.time() - t0),
      '\n**Sumber:** CIC-IDS2017 versi *GeneratedLabelledFlows* (memuat Source IP dan Timestamp), '
      'mirror Hugging Face `bencorn/CICIDS2017`, diverifikasi SHA-256.\n',
      '\n## 1. Bukti keaslian data\n',
      ('Setiap baris dicocokkan dengan versi *MachineLearningCVE* (versi 1):\n\n' +
       tabel(['Hari', 'Baris v2', 'Baris v1', 'Baris identik'],
             [(h, angka(a), angka(b), '%.2f%%' % (s * 100)) for h, a, b, s in cocok]) +
       '\nBaris dianggap identik bila semua nilai sama dengan toleransi pembulatan 10^-8 '
       '(berkas Senin versi 1 dibulatkan ke 10 desimal).\n') if cocok else
      'Pencocokan dengan versi 1 dilewati karena data versi 1 tidak tersedia di lingkungan ini '
      '(pada laptop peneliti hasilnya 100% identik untuk semua hari).\n',
      '\nBerkas Kamis pagi memuat %s baris kosong; baris ini dibuang.\n'
      % angka(sum(m - n for _, _, m, n in per_file)),
      '\n## 2. Perbaikan kolom waktu\n',
      '- Format 12 jam **tanpa AM/PM**; jam 1–7 adalah siang (13.00–19.00).\n',
      '- Senin tercatat sampai detik, Selasa–Jumat hanya sampai **menit**; semua waktu dibulatkan ke menit.\n\n',
      tabel(['Hari', 'Mulai', 'Selesai'], [(h, r['min'].strftime('%d-%m-%Y %H:%M'), r['max'].strftime('%H:%M'))
                                           for h, r in rentang.iterrows()]),
      '\n## 3. Fitur agregasi\nDihitung dari **seluruh %s aliran sebelum pembersihan** dengan fungsi yang sama '
      'dengan pipeline server (`04_Kode/nids-deteksi/nids/agregasi.py`). Jendela: %s detik.\n' % (
          angka(n_raw), ', '.join(map(str, JENDELA))),
      '\n## 4. Pembersihan\n\n',
      tabel(['Tahap', 'Sebelum', 'Sesudah', 'Dibuang', 'Keterangan'],
            [(t, angka(a), angka(b), angka(a - b), k) for t, a, b, k in langkah]),
      '\nVersi 1 membuang 307.068 baris sebagai duplikat karena hanya membandingkan fitur. '
      'Versi 2 hanya membuang baris yang seluruh kolomnya sama (termasuk Flow ID, port sumber, dan waktu).\n',
      '\n## 5. Jumlah per label: mentah vs versi 1 vs versi 2\n\n',
      tabel(['Label', 'Mentah', 'Versi 1', 'Versi 2'],
            [(lab, angka(r['Mentah']), angka(r['Versi 1']), angka(r['Versi 2']))
             for lab, r in tabel_label.sort_values('Mentah', ascending=False).iterrows()]),
      '\n## 6. Pembagian data\n\n',
      tabel(['Subset', 'Total', 'Normal', 'Serangan'],
            [(r.Subset, angka(r.Total), angka(r.Normal), angka(r.Serangan)) for r in pembagian.itertuples()]),
      '\n## 7. Catatan keterbatasan\n',
      '- %.1f%% aliran serangan CIC-IDS2017 berasal dari satu alamat IP (%s). Fitur agregasi per IP karenanya '
      'sangat membantu di dataset ini; pada server nyata penyerang dapat memakai banyak IP, sehingga agregasi '
      'per subnet /24 tetap disertakan.\n' % (penyerang.iloc[0] / penyerang.sum() * 100, penyerang.index[0]),
      '- Karena waktu hanya sampai menit, jendela agregasi pada CIC-IDS2017 bersifat per menit. '
      'Pada data server, waktu tersedia sampai mikrodetik.\n']
open(os.path.join(FOLDER, 'laporan_preprocessing_v2.md'), 'w', encoding='utf-8').write(''.join(md))
print('Selesai dalam %d detik. Laporan: laporan_preprocessing_v2.md' % (time.time() - t0))